In [1]:
import pytest
import numpy as np
import pandas as pd
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, numpy2ri
from rpy2.robjects.conversion import localconverter
from scipy.stats import norm
from survrm2py.rmst import exact_rmst1, rmst2reg, rmst, func_surv
from survrm2py.rmst_r import rmst_r

In [ ]:
ro.r("suppressPackageStartupMessages(library(survival))")
ro.r('data(pbc, package="survival")')

# 1. Get the R object
pbc_r = ro.globalenv["pbc"]

# 2. Force conversion to Pandas DataFrame
with localconverter(ro.default_converter + pandas2ri.converter + numpy2ri.converter) as cv:
    df = cv.rpy2py(pbc_r)
    # If it returns a recarray, convert it explicitly
    if isinstance(df, np.recarray):
        df = pd.DataFrame(df)

df = df.dropna(subset=["time", "status", "trt", "age", "bili", "protime"]).copy()
df["event"] = (df["status"] == 2).astype(int)
df["arm"] = (df["trt"] == 1).astype(int)


In [ ]:
    tau = min(df.groupby('arm')['time'].max())
    formula = "arm + " + " + ".join(covs)  # Translates to "arm + age"

    res_r = rmst_r(df, "time", "event", "arm", tau, formula="age")
    res_py = rmst(df, "time", "event", "arm", tau, formula="age")

    pd.testing.assert_frame_equal(
        res_py["adjusted_summary"].set_index("covariate"),
        res_r["adjusted_summary"].set_index("covariate"),
        check_dtype=False,
        atol=1e-10,
        rtol=1e-10,
    )


ValueError: All arrays must be of the same length

In [10]:
res_py["adjusted_summary"]

,covariate,coef,se(coef),z,p,lower .95,upper .95
0,Intercept,6705.911036,688.403892,9.741245,0.000000e+00,5356.664202,8055.157871
1,arm,444.616110,254.763123,1.745214,8.094767e-02,-54.710435,943.942655
2,age,-81.066260,12.382551,-6.546814,5.877743e-11,-105.335613,-56.796907


In [ ]:
%%timeit
res_r = rmst_r(df, "time", "event", "arm", tau, covariates=covs)

TypeError: rmst_r() got an unexpected keyword argument 'formula'

In [21]:
%%timeit
res_py = rmst(df, "time", "event", "arm", tau, covariates=covs)

9.14 ms ± 399 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [20]:
res_py['adjusted_summary']

,covariate,coef,se(coef),z,p,lower .95,upper .95
0,Intercept,6705.911036,688.403892,9.741245,0.000000e+00,5356.664202,8055.157871
1,arm,444.616110,254.763123,1.745214,8.094767e-02,-54.710435,943.942655
2,age,-81.066260,12.382551,-6.546814,5.877743e-11,-105.335613,-56.796907


In [22]:
res_py['adjusted_summary']

,covariate,coef,se(coef),z,p,lower .95,upper .95
0,Intercept,6705.911036,688.403892,9.741245,0.000000e+00,5356.664202,8055.157871
1,arm,444.616110,254.763123,1.745214,8.094767e-02,-54.710435,943.942655
2,age,-81.066260,12.382551,-6.546814,5.877743e-11,-105.335613,-56.796907
